In [1]:
"""
DEBUG SCRIPT 1: Analyze Raw YOLO Outputs
This script examines what YOLO actually outputs after export
"""

import tensorflow as tf
import numpy as np
import cv2
import glob
from ultralytics import YOLO

# ============================================================================
# PATHS
# ============================================================================
PT_MODEL = '/Users/rubenhayrapetyan/Downloads/Code/FRC/machine-learning/2025-Coral_Detection_Training/coral_detection/coral-detection-model-v111/weights/best.pt'
SAVED_MODEL = '/Users/rubenhayrapetyan/Downloads/Code/FRC/machine-learning/2025-Coral_Detection_Training/coral_detection/coral-detection-model-v111/weights/best_saved_model'
TEST_IMAGES = "/Users/rubenhayrapetyan/Downloads/Code/FRC/machine-learning/2025-Coral_Detection_Training/calib_images/*"

print("="*80)
print("DEBUG SCRIPT 1: RAW YOLO OUTPUT ANALYSIS")
print("="*80)

# ============================================================================
# Part 1: Get ground truth from original YOLO
# ============================================================================
print("\n[1] ORIGINAL YOLO (.pt) PREDICTIONS")
print("-"*80)

yolo = YOLO(PT_MODEL)
test_img = cv2.imread(glob.glob(TEST_IMAGES)[0])
img_resized = cv2.resize(test_img, (640, 640))

results = yolo.predict(img_resized, conf=0.01, verbose=False)[0]
print(f"\nNumber of detections: {len(results.boxes)}")

if len(results.boxes) > 0:
    for i, box in enumerate(results.boxes):
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].cpu().numpy()
        print(f"  Detection {i+1}:")
        print(f"    Confidence: {conf:.6f}")
        print(f"    Box (xyxy): {xyxy}")
else:
    print("  No detections!")

# ============================================================================
# Part 2: Examine raw SavedModel output
# ============================================================================
print("\n[2] RAW SAVEDMODEL OUTPUT")
print("-"*80)

model = tf.saved_model.load(SAVED_MODEL)
infer = model.signatures["serving_default"]

# Prepare input
img_float = img_resized.astype(np.float32) / 255.0
img_batch = np.expand_dims(img_float, axis=0)

# Get raw output
raw_out = infer(tf.constant(img_batch))
preds = list(raw_out.values())[0].numpy()  # [1, 6, 8400]

print(f"\nRaw output shape: {preds.shape}")
print(f"Raw output dtype: {preds.dtype}")

# Extract components
boxes_xywh = preds[0, 0:4, :]  # [4, 8400] - x, y, w, h
objectness = preds[0, 4, :]     # [8400] - objectness
class_scores = preds[0, 5:, :]  # [1, 8400] - class scores (if multi-class)

print(f"\n--- Box coordinates (xywh in pixels) ---")
print(f"X center range: [{boxes_xywh[0].min():.2f}, {boxes_xywh[0].max():.2f}]")
print(f"Y center range: [{boxes_xywh[1].min():.2f}, {boxes_xywh[1].max():.2f}]")
print(f"Width range: [{boxes_xywh[2].min():.2f}, {boxes_xywh[2].max():.2f}]")
print(f"Height range: [{boxes_xywh[3].min():.2f}, {boxes_xywh[3].max():.2f}]")

print(f"\n--- Objectness scores ---")
print(f"Range: [{objectness.min():.6f}, {objectness.max():.6f}]")
print(f"Mean: {objectness.mean():.6f}")
print(f"Median: {np.median(objectness):.6f}")
print(f"Std dev: {objectness.std():.6f}")
print(f"Top 10 values: {sorted(objectness, reverse=True)[:10]}")

# Find the anchor with highest objectness
max_idx = np.argmax(objectness)
print(f"\n--- Best anchor (index {max_idx}) ---")
print(f"Objectness: {objectness[max_idx]:.6f}")
print(f"Box (xywh): {boxes_xywh[:, max_idx]}")
if class_scores.size > 0:
    print(f"Class score: {class_scores[0, max_idx]:.6f}")

# ============================================================================
# Part 3: Test different score interpretations
# ============================================================================
print("\n[3] TESTING SCORE INTERPRETATIONS")
print("-"*80)

# Hypothesis 1: Scores are already probabilities (0-1)
print("\nHypothesis 1: Scores are already probabilities")
print(f"  Values > 0.9: {np.sum(objectness > 0.9)}")
print(f"  Values > 0.5: {np.sum(objectness > 0.5)}")
print(f"  Values > 0.25: {np.sum(objectness > 0.25)}")

# Hypothesis 2: Scores need sigmoid
objectness_sigmoid = 1 / (1 + np.exp(-objectness))
print("\nHypothesis 2: After applying sigmoid")
print(f"  Range: [{objectness_sigmoid.min():.6f}, {objectness_sigmoid.max():.6f}]")
print(f"  Values > 0.9: {np.sum(objectness_sigmoid > 0.9)}")
print(f"  Values > 0.5: {np.sum(objectness_sigmoid > 0.5)}")
print(f"  Top 10: {sorted(objectness_sigmoid, reverse=True)[:10]}")

# Hypothesis 3: Need to multiply objectness * class_score
if class_scores.size > 0:
    print("\nHypothesis 3: Multiply objectness * class_score")
    combined = objectness * class_scores[0]
    print(f"  Range: [{combined.min():.6f}, {combined.max():.6f}]")
    print(f"  Values > 0.5: {np.sum(combined > 0.5)}")
    print(f"  Values > 0.25: {np.sum(combined > 0.25)}")
    print(f"  Top 10: {sorted(combined, reverse=True)[:10]}")
    
    # Find best combined score
    max_combined_idx = np.argmax(combined)
    print(f"\n  Best combined score anchor (index {max_combined_idx}):")
    print(f"    Objectness: {objectness[max_combined_idx]:.6f}")
    print(f"    Class score: {class_scores[0, max_combined_idx]:.6f}")
    print(f"    Combined: {combined[max_combined_idx]:.6f}")
    print(f"    Box (xywh): {boxes_xywh[:, max_combined_idx]}")

# ============================================================================
# Part 4: Compare best anchor with YOLO detection
# ============================================================================
print("\n[4] COMPARING WITH GROUND TRUTH")
print("-"*80)

if len(results.boxes) > 0:
    # Get YOLO's best detection
    yolo_conf = float(results.boxes[0].conf[0])
    yolo_box = results.boxes[0].xyxy[0].cpu().numpy()
    
    # Convert to xywh
    yolo_xywh = [
        (yolo_box[0] + yolo_box[2]) / 2,  # center x
        (yolo_box[1] + yolo_box[3]) / 2,  # center y
        yolo_box[2] - yolo_box[0],         # width
        yolo_box[3] - yolo_box[1]          # height
    ]
    
    print(f"\nYOLO detection:")
    print(f"  Confidence: {yolo_conf:.6f}")
    print(f"  Box (xywh): {yolo_xywh}")
    
    print(f"\nBest raw anchor (by objectness):")
    print(f"  Objectness: {objectness[max_idx]:.6f}")
    print(f"  Box (xywh): {boxes_xywh[:, max_idx]}")
    
    # Calculate distance
    dist = np.linalg.norm(boxes_xywh[:, max_idx] - yolo_xywh)
    print(f"\n  Distance between boxes: {dist:.2f} pixels")
    
    if dist < 50:
        print("  ✅ Boxes match! This anchor corresponds to YOLO's detection")
    else:
        print("  ⚠️  Boxes don't match well")

# ============================================================================
# Part 5: Recommendations
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)

print("\nBased on the objectness range:")
if objectness.max() <= 1.0:
    print("✅ Objectness is in [0,1] - ALREADY SIGMOID'D")
    print("   Do NOT apply sigmoid in the wrapper!")
else:
    print("⚠️  Objectness > 1.0 - NEEDS SIGMOID")
    print("   Apply sigmoid in the wrapper")

print("\nBased on score distribution:")
high_obj = np.sum(objectness > 0.5)
if high_obj > 100:
    print(f"⚠️  {high_obj} anchors have objectness > 0.5")
    print("   This is too many - scores might be wrong")
elif high_obj > 0:
    print(f"✅ {high_obj} anchors have objectness > 0.5")
    print("   This looks reasonable")
else:
    print("⚠️  No anchors with objectness > 0.5")
    print("   Check if multiplication with class scores is needed")

print("\n" + "="*80)
print("Copy the output above and we'll fix the wrapper based on this!")
print("="*80)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

DEBUG SCRIPT 1: RAW YOLO OUTPUT ANALYSIS

[1] ORIGINAL YOLO (.pt) PREDICTIONS
--------------------------------------------------------------------------------

Number of detections: 1
  Detection 1:
    Confidence: 0.980668
    Box (xyxy): [     175.11      45.998       413.4      292.95]

[2] RAW SAVEDMODEL OUTPUT
--------------------------------------------------------------------------------

Raw output shape: (1, 6, 8400)
Raw output dtype: float32

--- Box coordinates (xywh in pixels) ---
X center range: [2.87, 637.49]
Y center range: [3.01, 637.49]
Width range: [5.46, 543.19]
Height range: [5.29, 530.14]

--- Objectness scores ---
Range: [0.000000, 0.892841]
Mean: 0.000830
Median: 0.000000
Std dev: 0.023475
Top 10 values: [0.89284116, 0.806321, 0.7964221, 0.75730026, 0.75626045, 0.6600333, 0.6275808, 0.53951854, 0.3640693, 0.35796404]

--- Best anchor (index 8089) ---
Objectness: 0.892841
Box (xywh): [     294.66      169.71       238.6      247.43]
Class score: 0.001450

[3] TEST

In [2]:
"""
DEBUG SCRIPT 2: Manually Test NMS Logic
This script tests the NMS wrapper step-by-step
"""

import tensorflow as tf
import numpy as np
import cv2
import glob

# ============================================================================
# PATHS
# ============================================================================
SAVED_MODEL = '/Users/rubenhayrapetyan/Downloads/Code/FRC/machine-learning/2025-Coral_Detection_Training/coral_detection/coral-detection-model-v111/weights/best_saved_model'
TEST_IMAGES = "/Users/rubenhayrapetyan/Downloads/Code/FRC/machine-learning/2025-Coral_Detection_Training/calib_images/*"

print("="*80)
print("DEBUG SCRIPT 2: MANUAL NMS WRAPPER TEST")
print("="*80)

# Load model
model = tf.saved_model.load(SAVED_MODEL)
infer = model.signatures["serving_default"]

# Load test image
test_img = cv2.imread(glob.glob(TEST_IMAGES)[0])
img_resized = cv2.resize(test_img, (640, 640))
img_float = img_resized.astype(np.float32) / 255.0
img_batch = np.expand_dims(img_float, axis=0)

# Get raw predictions
print("\n[1] Getting raw YOLO predictions...")
raw_out = infer(tf.constant(img_batch))
preds = list(raw_out.values())[0]  # [1, 6, 8400]
print(f"Raw predictions shape: {preds.shape}")

# ============================================================================
# Step-by-step NMS processing
# ============================================================================

# Step 1: Extract boxes and scores
print("\n[2] Extracting boxes and scores...")
boxes_xywh = preds[:, 0:4, :]    # [1, 4, 8400]
objectness = preds[:, 4:5, :]     # [1, 1, 8400]

# Check for class scores
num_classes = preds.shape[1] - 5
print(f"Number of classes: {num_classes}")

# Transpose
boxes_xywh = tf.transpose(boxes_xywh, [0, 2, 1])  # [1, 8400, 4]
objectness = tf.transpose(objectness, [0, 2, 1])   # [1, 8400, 1]

print(f"Boxes shape: {boxes_xywh.shape}")
print(f"Objectness shape: {objectness.shape}")

# Step 2: Calculate scores (TEST MULTIPLE METHODS)
print("\n[3] Testing different score calculations...")

# Method A: Just use objectness (no sigmoid)
scores_a = objectness
print(f"\nMethod A (raw objectness):")
print(f"  Range: [{tf.reduce_min(scores_a):.6f}, {tf.reduce_max(scores_a):.6f}]")
print(f"  > 0.5: {tf.reduce_sum(tf.cast(scores_a > 0.5, tf.int32)).numpy()}")
print(f"  > 0.25: {tf.reduce_sum(tf.cast(scores_a > 0.25, tf.int32)).numpy()}")

# Method B: Apply sigmoid
scores_b = tf.sigmoid(objectness)
print(f"\nMethod B (sigmoid applied):")
print(f"  Range: [{tf.reduce_min(scores_b):.6f}, {tf.reduce_max(scores_b):.6f}]")
print(f"  > 0.5: {tf.reduce_sum(tf.cast(scores_b > 0.5, tf.int32)).numpy()}")
print(f"  > 0.25: {tf.reduce_sum(tf.cast(scores_b > 0.25, tf.int32)).numpy()}")

# Method C: Multiply with class scores (if exists)
if num_classes > 0:
    class_scores = preds[:, 5:, :]
    class_scores = tf.transpose(class_scores, [0, 2, 1])
    max_class = tf.reduce_max(class_scores, axis=-1, keepdims=True)
    scores_c = objectness * max_class
    print(f"\nMethod C (objectness * class_score):")
    print(f"  Range: [{tf.reduce_min(scores_c):.6f}, {tf.reduce_max(scores_c):.6f}]")
    print(f"  > 0.5: {tf.reduce_sum(tf.cast(scores_c > 0.5, tf.int32)).numpy()}")
    print(f"  > 0.25: {tf.reduce_sum(tf.cast(scores_c > 0.25, tf.int32)).numpy()}")
    
    # Method D: Both sigmoid
    scores_d = tf.sigmoid(objectness) * tf.sigmoid(max_class)
    print(f"\nMethod D (sigmoid both):")
    print(f"  Range: [{tf.reduce_min(scores_d):.6f}, {tf.reduce_max(scores_d):.6f}]")
    print(f"  > 0.5: {tf.reduce_sum(tf.cast(scores_d > 0.5, tf.int32)).numpy()}")
    print(f"  > 0.25: {tf.reduce_sum(tf.cast(scores_d > 0.25, tf.int32)).numpy()}")
else:
    scores_c = None
    scores_d = None

# Pick the best method based on analysis
print("\n" + "="*80)
print("Which method gives reasonable scores (a few detections > 0.5)?")
print("="*80)

# For now, let's test with Method A (raw objectness)
USE_SCORES = scores_a
print(f"\nUsing Method A for NMS test...")

# Step 3: Convert boxes to normalized format
print("\n[4] Converting boxes to normalized format...")

cx = boxes_xywh[..., 0:1]
cy = boxes_xywh[..., 1:2]
w = boxes_xywh[..., 2:3]
h = boxes_xywh[..., 3:4]

# Convert to corners
x1 = cx - w / 2.0
y1 = cy - h / 2.0
x2 = cx + w / 2.0
y2 = cy + h / 2.0

# Normalize
x1_norm = tf.clip_by_value(x1 / 640.0, 0.0, 1.0)
y1_norm = tf.clip_by_value(y1 / 640.0, 0.0, 1.0)
x2_norm = tf.clip_by_value(x2 / 640.0, 0.0, 1.0)
y2_norm = tf.clip_by_value(y2 / 640.0, 0.0, 1.0)

# TF format: [y1, x1, y2, x2]
boxes_norm = tf.concat([y1_norm, x1_norm, y2_norm, x2_norm], axis=-1)
boxes_for_nms = tf.expand_dims(boxes_norm, axis=2)  # [1, 8400, 1, 4]

print(f"Normalized boxes shape: {boxes_for_nms.shape}")
print(f"Box range: [{tf.reduce_min(boxes_for_nms):.4f}, {tf.reduce_max(boxes_for_nms):.4f}]")

# Step 4: Apply NMS
print("\n[5] Applying NMS...")

nms_result = tf.image.combined_non_max_suppression(
    boxes=boxes_for_nms,
    scores=USE_SCORES,
    max_output_size_per_class=10,
    max_total_size=10,
    iou_threshold=0.45,
    score_threshold=0.25,
    clip_boxes=False
)

print(f"\nNMS Results:")
print(f"  valid_detections: {nms_result.valid_detections.numpy()}")
print(f"  nmsed_scores shape: {nms_result.nmsed_scores.shape}")
print(f"  nmsed_scores: {nms_result.nmsed_scores.numpy()[0]}")
print(f"  nmsed_boxes shape: {nms_result.nmsed_boxes.shape}")

# Step 5: Test with different thresholds
print("\n[6] Testing different NMS thresholds...")

for threshold in [0.01, 0.1, 0.25, 0.5]:
    nms_test = tf.image.combined_non_max_suppression(
        boxes=boxes_for_nms,
        scores=USE_SCORES,
        max_output_size_per_class=10,
        max_total_size=10,
        iou_threshold=0.45,
        score_threshold=threshold,
        clip_boxes=False
    )
    num_dets = int(nms_test.valid_detections.numpy()[0])
    max_score = nms_test.nmsed_scores.numpy()[0].max()
    print(f"  Threshold {threshold}: {num_dets} detections, max score: {max_score:.6f}")

# ============================================================================
# Try all methods with low threshold
# ============================================================================
print("\n[7] Testing all score methods with threshold=0.01...")

for method_name, scores_method in [
    ("A: Raw objectness", scores_a),
    ("B: Sigmoid objectness", scores_b),
    ("C: Obj * class", scores_c),
    ("D: Sigmoid both", scores_d)
]:
    if scores_method is None:
        continue
    
    nms_test = tf.image.combined_non_max_suppression(
        boxes=boxes_for_nms,
        scores=scores_method,
        max_output_size_per_class=10,
        max_total_size=10,
        iou_threshold=0.45,
        score_threshold=0.01,
        clip_boxes=False
    )
    
    num_dets = int(nms_test.valid_detections.numpy()[0])
    scores_out = nms_test.nmsed_scores.numpy()[0]
    max_score = scores_out.max()
    
    print(f"\n{method_name}:")
    print(f"  Detections: {num_dets}")
    print(f"  Max score: {max_score:.6f}")
    if num_dets > 0:
        print(f"  Scores: {scores_out[:num_dets]}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print("\nWhich method produced the best results?")
print("Expected: ~1 detection with score ~0.9")
print("\nShare this output and we'll fix the wrapper!")
print("="*80)

DEBUG SCRIPT 2: MANUAL NMS WRAPPER TEST

[1] Getting raw YOLO predictions...
Raw predictions shape: (1, 6, 8400)

[2] Extracting boxes and scores...
Number of classes: 1
Boxes shape: (1, 8400, 4)
Objectness shape: (1, 8400, 1)

[3] Testing different score calculations...

Method A (raw objectness):
  Range: [0.000000, 0.892841]
  > 0.5: 8
  > 0.25: 10

Method B (sigmoid applied):
  Range: [0.500000, 0.709476]
  > 0.5: 4324
  > 0.25: 8400

Method C (objectness * class_score):
  Range: [0.000000, 0.008310]
  > 0.5: 0
  > 0.25: 0

Method D (sigmoid both):
  Range: [0.250000, 0.354995]
  > 0.5: 0
  > 0.25: 6208

Which method gives reasonable scores (a few detections > 0.5)?

Using Method A for NMS test...

[4] Converting boxes to normalized format...
Normalized boxes shape: (1, 8400, 1, 4)
Box range: [0.0000, 1.0000]

[5] Applying NMS...

NMS Results:
  valid_detections: [1]
  nmsed_scores shape: (1, 10)
  nmsed_scores: [    0.89284           0           0           0           0          